# 02 - Exploratory Visualisation

**Purpose:** Visualise the time series data to:
1. Understand trends and seasonality in each indicator
2. Assess temporal alignment between economic shocks and food insecurity
3. Identify candidate leading indicators for lag analysis

**Key Question:** Is there sufficient temporal overlap to perform meaningful cross-correlation?

In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime
import numpy as np

# Configure matplotlib for clean output
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

print(f"Loading data from: {PROCESSED_DATA}")

## 1. Load Processed Data

In [ ]:
# Load all processed datasets
fuel_poverty = pl.read_parquet(PROCESSED_DATA / "fuel_poverty.parquet")
disposable_income = pl.read_parquet(PROCESSED_DATA / "disposable_income.parquet")
absolute_poverty = pl.read_parquet(PROCESSED_DATA / "absolute_poverty.parquet")
food_insecurity = pl.read_parquet(PROCESSED_DATA / "food_insecurity.parquet")

print("Loaded datasets:")
print(f"  - Fuel poverty: {fuel_poverty.shape}")
print(f"  - Disposable income: {disposable_income.shape}")
print(f"  - Absolute poverty: {absolute_poverty.shape}")
print(f"  - Food insecurity: {food_insecurity.shape}")

## 2. Economic Fairness Indicators (Shock Signals)

### 2.1 Fuel Poverty Trend

Fuel poverty measures the proportion of households spending a high share of income on energy.
This is a direct stress indicator — households in fuel poverty often face food/fuel trade-offs.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

fp_df = fuel_poverty.to_pandas()

ax.plot(fp_df['year'], fp_df['london_pct'], 'o-', color='#E63946', 
        linewidth=2, markersize=6, label='London')
ax.plot(fp_df['year'], fp_df['england_pct'], 's--', color='#457B9D', 
        linewidth=2, markersize=5, label='England')

ax.set_xlabel('Year')
ax.set_ylabel('Households in Fuel Poverty (%)')
ax.set_title('Fuel Poverty Rate: London vs England (2003-2018)')
ax.legend()
ax.set_ylim(0, max(fp_df['england_pct'].max(), fp_df['london_pct'].max()) * 1.15)

# Annotate key events
ax.axvline(x=2008, color='gray', linestyle=':', alpha=0.7)
ax.text(2008.1, ax.get_ylim()[1]*0.9, '2008\nFinancial\nCrisis', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(OUTPUTS / 'fuel_poverty_trend.png', dpi=150)
plt.show()

print(f"\nFuel poverty range: {fp_df['london_pct'].min():.1f}% - {fp_df['london_pct'].max():.1f}% (London)")

### 2.2 Disposable Income Trend

Median equivalised weekly disposable income (After Housing Costs) indicates household spending capacity.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

inc_df = disposable_income.to_pandas()

# Extract mid-year from period string for x-axis
inc_df['mid_year'] = inc_df['period'].apply(lambda x: int(x.split('/')[0].split('-')[0]) + 1 if isinstance(x, str) else None)
inc_df = inc_df.dropna(subset=['mid_year'])

ax.plot(inc_df['mid_year'], inc_df['london'], 'o-', color='#E63946', 
        linewidth=2, markersize=6, label='London')
ax.plot(inc_df['mid_year'], inc_df['uk'], 's--', color='#457B9D', 
        linewidth=2, markersize=5, label='UK')

ax.set_xlabel('Year (mid-point of 3-year period)')
ax.set_ylabel('Weekly Disposable Income (£, 2016 prices)')
ax.set_title('Median Equivalised Disposable Income AHC (3-year rolling average)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUTS / 'disposable_income_trend.png', dpi=150)
plt.show()

print(f"\nLondon income range: £{inc_df['london'].min():.0f} - £{inc_df['london'].max():.0f} per week")

### 2.3 Absolute Poverty Trend

Absolute poverty (income below 60% of 2010/11 median) shows the proportion of the population in material hardship.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

pov_df = absolute_poverty.to_pandas()

# Extract end year from period (e.g., "94/95-96/97" -> 1997)
def extract_end_year(period):
    try:
        end_part = period.split('-')[1]
        year = int(end_part.split('/')[1])
        return 1900 + year if year > 50 else 2000 + year
    except:
        return None

pov_df['end_year'] = pov_df['period'].apply(extract_end_year)
pov_df = pov_df.dropna(subset=['end_year'])

ax.plot(pov_df['end_year'], pov_df['london_ahc_pct'], 'o-', color='#E63946', 
        linewidth=2, markersize=5, label='London (AHC)')
ax.plot(pov_df['end_year'], pov_df['london_bhc_pct'], 's--', color='#A8DADC', 
        linewidth=1.5, markersize=4, alpha=0.7, label='London (BHC)')

ax.set_xlabel('Year (end of 3-year period)')
ax.set_ylabel('Population in Absolute Poverty (%)')
ax.set_title('Absolute Poverty Rate - London')
ax.legend()

# Annotate
ax.axvline(x=2008, color='gray', linestyle=':', alpha=0.7)
ax.text(2008.2, ax.get_ylim()[1]*0.85, '2008 Crisis', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(OUTPUTS / 'absolute_poverty_trend.png', dpi=150)
plt.show()

print(f"\nAbsolute poverty (AHC) range: {pov_df['london_ahc_pct'].min():.0f}% - {pov_df['london_ahc_pct'].max():.0f}%")

## 3. Food Insecurity (Response Signal)

The Food Foundation tracker captures the pandemic and cost-of-living crisis period.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

fi_df = food_insecurity.to_pandas()
fi_df['date'] = pd.to_datetime(fi_df['survey_date'])

ax.plot(fi_df['date'], fi_df['pct_food_insecure_all'], 'o-', color='#E63946', 
        linewidth=2, markersize=8, label='All households')

# Plot children data where available
fi_children = fi_df.dropna(subset=['pct_food_insecure_children'])
if len(fi_children) > 0:
    ax.plot(fi_children['date'], fi_children['pct_food_insecure_children'], 
            's-', color='#F4A261', linewidth=2, markersize=7, label='Households with children')

ax.set_xlabel('Survey Date')
ax.set_ylabel('Households Food Insecure (%)')
ax.set_title('UK Food Insecurity Rate (Food Foundation / FSA Tracker)')
ax.legend()

# Format x-axis dates
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)

# Annotate key events
ax.axvline(x=pd.Timestamp('2020-03-23'), color='gray', linestyle=':', alpha=0.7)
ax.text(pd.Timestamp('2020-04-15'), ax.get_ylim()[1]*0.95, 'First\nLockdown', fontsize=9, color='gray', ha='left')

ax.axvline(x=pd.Timestamp('2022-04-01'), color='gray', linestyle=':', alpha=0.7)
ax.text(pd.Timestamp('2022-04-15'), ax.get_ylim()[1]*0.85, 'Energy\nPrice Cap\nRise', fontsize=9, color='gray', ha='left')

plt.tight_layout()
plt.savefig(OUTPUTS / 'food_insecurity_trend.png', dpi=150)
plt.show()

print(f"\nFood insecurity range: {fi_df['pct_food_insecure_all'].min():.1f}% - {fi_df['pct_food_insecure_all'].max():.1f}%")
print(f"Peak: {fi_df.loc[fi_df['pct_food_insecure_all'].idxmax(), 'survey_date']} at {fi_df['pct_food_insecure_all'].max():.1f}%")

## 4. Temporal Alignment Analysis

Let's visualise all time series on a common timeline to assess overlap.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Convert all to datetime for common x-axis
fp_df['date'] = pd.to_datetime(fp_df['year'], format='%Y')
inc_df['date'] = pd.to_datetime(inc_df['mid_year'].astype(int), format='%Y')
pov_df['date'] = pd.to_datetime(pov_df['end_year'].astype(int), format='%Y')

# Panel 1: Fuel Poverty
axes[0].plot(fp_df['date'], fp_df['london_pct'], 'o-', color='#E63946', linewidth=2, markersize=5)
axes[0].set_ylabel('Fuel Poverty\n(%)')
axes[0].set_title('Economic Stress Indicators vs Food Insecurity: Temporal Coverage', fontsize=14)
axes[0].fill_between(fp_df['date'], 0, fp_df['london_pct'], alpha=0.3, color='#E63946')

# Panel 2: Disposable Income (inverted - lower is worse)
axes[1].plot(inc_df['date'], inc_df['london'], 'o-', color='#2A9D8F', linewidth=2, markersize=5)
axes[1].set_ylabel('Disposable\nIncome (£/wk)')
axes[1].fill_between(inc_df['date'], inc_df['london'].min(), inc_df['london'], alpha=0.3, color='#2A9D8F')

# Panel 3: Absolute Poverty
axes[2].plot(pov_df['date'], pov_df['london_ahc_pct'], 'o-', color='#F4A261', linewidth=2, markersize=5)
axes[2].set_ylabel('Absolute\nPoverty (%)')
axes[2].fill_between(pov_df['date'], 0, pov_df['london_ahc_pct'], alpha=0.3, color='#F4A261')

# Panel 4: Food Insecurity
axes[3].plot(fi_df['date'], fi_df['pct_food_insecure_all'], 'o-', color='#264653', linewidth=2, markersize=6)
axes[3].set_ylabel('Food\nInsecurity (%)')
axes[3].set_xlabel('Year')
axes[3].fill_between(fi_df['date'], 0, fi_df['pct_food_insecure_all'], alpha=0.3, color='#264653')

# Add gap annotation
gap_start = pd.Timestamp('2018-01-01')
gap_end = pd.Timestamp('2020-03-01')
for ax in axes:
    ax.axvspan(gap_start, gap_end, alpha=0.15, color='red', label='Data gap')

# Format x-axis
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[3].xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig(OUTPUTS / 'temporal_alignment.png', dpi=150)
plt.show()

print("\n" + "="*60)
print("TEMPORAL COVERAGE SUMMARY")
print("="*60)
print(f"Fuel Poverty:     {fp_df['date'].min().year} - {fp_df['date'].max().year}")
print(f"Disposable Income: {inc_df['date'].min().year} - {inc_df['date'].max().year}")
print(f"Absolute Poverty:  {pov_df['date'].min().year} - {pov_df['date'].max().year}")
print(f"Food Insecurity:   {fi_df['date'].min().strftime('%Y-%m')} - {fi_df['date'].max().strftime('%Y-%m')}")
print("\n*** GAP: Economic indicators end ~2018, Food insecurity starts 2020 ***")

## 5. Key Observations

### Patterns Identified

1. **Fuel Poverty**: Relatively stable 8-11% in London, with a notable spike around 2006-2007

2. **Disposable Income**: Declined 2010-2014 (austerity), recovered 2015-2018

3. **Absolute Poverty**: Dramatic long-term decline from ~40% (1997) to ~15% (2018)

4. **Food Insecurity**: 
   - Pandemic spike (16%) → rapid drop to 7% by summer 2020
   - Cost-of-living crisis drove peak to 18.4% (Sept 2022)
   - FSA measure shows higher baseline (25%) — different methodology

### Temporal Gap Challenge

The ~2 year gap between Economic Fairness data (ends 2018) and Food Foundation tracking (starts March 2020) prevents direct lag analysis with these specific datasets.

### Recommended Next Steps

1. **Supplement with contemporary economic data:**
   - ONS Claimant Count (monthly, current)
   - BEIS/DESNZ energy price statistics
   - Bank of England inflation data

2. **Alternative analysis approach:**
   - Use pre-pandemic economic levels as "vulnerability baseline"
   - Correlate Food Insecurity with concurrent economic shocks (energy prices, benefits changes)

3. **For the policy report:**
   - Frame historical economic data as context for "who is vulnerable"
   - Use Food Foundation time series to demonstrate response dynamics

In [ ]:
# Summary statistics for report
print("SUMMARY STATISTICS FOR POLICY REPORT")
print("="*60)
print("\nFood Insecurity Dynamics (2020-2025):")
print(f"  - Initial pandemic spike: 16% (March 2020)")
print(f"  - Post-lockdown low: 7% (August 2020)")
print(f"  - Cost-of-living peak: 18.4% (September 2022)")
print(f"  - Current level: 14% (January 2025)")
print(f"\n  Implied response time to shocks: ~3-6 months (visual inspection)")
print("\nPre-pandemic Vulnerability Context (London):")
print(f"  - Fuel poverty: ~10% of households (2018)")
print(f"  - Absolute poverty (AHC): ~22% of population (2018)")
print(f"  - Median disposable income: £418/week (2017-18)")

## 6. Save Outputs

Export figures and summary data for the policy report.

In [ ]:
print("Saved outputs:")
for f in OUTPUTS.glob("*.png"):
    print(f"  - {f.name}")

print("\nNext notebook: 03_correlation_study.ipynb")
print("  - Will require supplementary economic data with 2020+ coverage")
print("  - Or: Analyse Food Insecurity series alone for response dynamics")